# Amazon Nova RFT Multiturn - Quick Start Guide

This notebook demonstrates RFT (Reinforcement Fine-Tuning) multiturn training and evaluation for multi-turn conversational tasks.

## Table of Contents

- [Step 1: Install and Import SDK](#step-1-install-and-import-sdk)
- [Step 1a: Verify Nova Forge Access via Starter Kit](#step-1a-verify-nova-forge-access-via-starter-kit)
- [Step 1b: Create Custom Environment (Optional)](#step-1b-create-custom-environment-optional)
- [Step 2: Configure Your AWS Resources](#step-2-configure-your-aws-resources)
  - [Step 2a: Create RFT Execution Role (Optional)](#step-2a-optional-create-rft-execution-role)
  - [Step 2b: List Existing Stacks (Optional)](#step-2b-optional-list-existing-cloudformation-stacks)
- [Step 3: Setup RFT Multiturn Infrastructure](#step-3-setup-rft-multiturn-infrastructure)
- [Step 4: Deploy Infrastructure](#step-4-deploy-infrastructure)
- [Step 5: Start Training Environment](#step-5-start-training-environment)
- [Step 6: Configure Training](#step-6-configure-training)
- [Step 7: Start Training](#step-7-start-training)
- [Step 8: Monitor Training](#step-8-monitor-training)
- [Step 9: Evaluation](#step-9-evaluation)
- [Step 10: Cleanup](#step-10-cleanup)

## What You'll Learn

1. Setting up RFT multiturn infrastructure (LOCAL/EC2/ECS)
2. Preparing multi-turn conversation data
3. Training with RFT multiturn on HyperPod
4. Evaluating trained models
5. Monitoring and cleanup

## Prerequisites

- AWS credentials configured
- The SDK requires specific IAM permissions. See the [IAM Roles/Policies section in the main README](../README.md#iam-rolespolicies) for the complete list of required permissions. **For RFT Multiturn**, additional SSM and ECS permissions are required - see the "If performing RFT Multiturn training" section in the README.
- S3 bucket for data and artifacts
- SageMaker HyperPod cluster (RFT multiturn only supports SMHP)
- Nova Customization SDK installed
- HyperPod CLI installed (see SDK README)
- Nova Forge subscription enabled

### What This Notebook Does

This notebook will:

1. **Setup Infrastructure**: Deploy Lambda functions, SQS queues, and reward workers
   - Creates IAM execution role with required permissions (CloudFormation, Lambda, SQS, DynamoDB, S3, CloudWatch)
   - Deploys CloudFormation stack via SAM (Serverless Application Model)

2. **Configure Environment**: Install starter kit and custom reward functions
   - Downloads Nova RFT starter kit from S3 to your chosen platform (LOCAL/EC2/ECS)
   - Creates Python virtual environment at specified location
   - Installs dependencies

3. **Start Training**: Launch RFT training job on HyperPod with reward feedback
   - Starts reward worker processes on your chosen platform
   - Submits training job to SageMaker HyperPod cluster

4. **Monitor Progress**: Track training metrics and reward scores
   - View CloudWatch logs from training and training/evaluation environment

5. **Evaluate**: Test the trained model on your data
   - Stops training workers and starts evaluation workers
   - Runs evaluation on test dataset with reward computation
   - Generates evaluation metrics and results

6. **Cleanup**: Remove infrastructure and stop workers
   - Stops reward worker processes
   - Optionally deletes CloudFormation stack and all resources
   - Optionally cleans up virtual environment and downloaded files

## Step 1: Install and Import SDK

In [ ]:
!cd ../ && pip install .

In [ ]:
import os

import boto3
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound


def load_credentials(profile=None):
    """Load AWS credentials with fallback behavior."""
    if profile:
        try:
            session = boto3.Session(profile_name=profile)
            credentials = session.get_credentials()
            if not credentials:
                raise RuntimeError(f"No credentials found for profile '{profile}'")
        except ProfileNotFound:
            raise RuntimeError(f"Profile '{profile}' not found in credentials file")
    else:
        try:
            session = boto3.Session()
            credentials = session.get_credentials()
            if not credentials:
                raise RuntimeError("No credentials found in current AWS session")
        except NoCredentialsError:
            raise RuntimeError("No AWS credentials configured")

    # Validate credentials
    try:
        sts_client = session.client("sts")
        sts_client.get_caller_identity()
    except ClientError as e:
        raise RuntimeError(f"Invalid AWS credentials: {e}")

    return {
        "aws_access_key_id": credentials.access_key,
        "aws_secret_access_key": credentials.secret_key,
        "aws_session_token": credentials.token,
        "region_name": session.region_name or "us-east-1",
    }

In [ ]:
creds = load_credentials()

In [ ]:
import time

from amzn_nova_customization_sdk import *

print("✅ SDK imported successfully!")

## Step 1a: Verify Starter Kit Access

Verify that you have access to the RFT starter kit S3 bucket. This is required for setting up the infrastructure.

In [ ]:
# Check access to starter kit bucket
!aws s3 ls s3://nova-rft-starter-kit-c7363-206080352451-us-east-1/v1/ --region us-east-1

# Ensure your IAM role has the following permissions:
# - s3:ListBucket on arn:aws:s3:::nova-rft-starter-kit-*
# - s3:GetObject on arn:aws:s3:::nova-rft-starter-kit-*/*

## Step 1b: Create Custom Environment (Optional)

For RFT multiturn, you can use built-in environments (wordle, terminalbench) or create your own.

In [ ]:
# Create a custom environment
custom_env = CustomEnvironment(
    env_id="my-custom-env", output_dir="~/custom_envs/", env_type="single_turn"
).create(overwrite=True)

# # Load an existing environment
# custom_env = CustomEnvironment(
#     env_id="my-custom-env",
#     local_path="~/custom_envs/",
#     env_type="single_turn"
# ).load()


print("✅ Custom environment created")

In [ ]:
# Validate and upload to S3
custom_env.validate()
custom_env.package_and_upload()

print(f"✅ Custom environment uploaded: {custom_env.s3_uri}")

## Step 2: Configure Your AWS Resources

In [ ]:
# TODO: Update these values for your environment
S3_BUCKET = "nova-customization-beta"  # TODO: Replace with your S3 bucket
S3_DATA_PATH = f"s3://{S3_BUCKET}/rft-multiturn/input"
S3_OUTPUT_PATH = f"s3://{S3_BUCKET}/rft-multiturn/output"

# RFT Infrastructure Configuration
STACK_NAME = "<sample-stack-name>"  # CloudFormation stack name: Note NovaSDK suffix will be added to the name for resource management
REGION = "us-east-1"
VF_ENV_ID = "wordle"  # Environment: wordle, terminalbench_env, etc.
PYTHON_VENV_NAME = (
    "my_rft_venv"  # Python virtual environment name (required for LOCAL/EC2)
)
# HyperPod Configuration
HYPERPOD_CLUSTER = "<cluster-name>"  # TODO: Replace with your cluster name
NAMESPACE = "<namespace>"  # TODO: Replace with your namespace

print(f"Data Path: {S3_DATA_PATH}")
print(f"Output Path: {S3_OUTPUT_PATH}")

## Step 2a: (Optional) Create RFT Execution Role

The SDK can automatically create an IAM role with required permissions for RFT infrastructure.
If you already have a role, skip this step and use your existing role name.

In [ ]:
# Create role with default name (RFTExecutionRoleNovaSDK)
role_arn = create_rft_execution_role(region=REGION)
print(f"Created role: {role_arn}")

# Or create with custom name
# role_arn = create_rft_execution_role(region=REGION, role_name="my-custom-rft-role")

# Or pass with custom name and policy json
# role_arn = create_rft_execution_role(region=REGION, role_name="my-custom-rft-role, custom_policy_path=<path to custom policy json>")

# Extract role name for later use
RFT_ROLE_NAME = role_arn.split("/")[-1]
print(f"Role name: {RFT_ROLE_NAME}")

## Step 2b: (Optional) List Existing CloudFormation Stacks

Check if you have existing RFT stacks to avoid naming conflicts or reuse existing infrastructure.

In [ ]:
# List all RFT stacks in the region
stacks = list_rft_stacks(region=REGION)

print(f"Found {len(stacks)} RFT stack(s):")
for stack_name in stacks:
    print(f"  - {stack_name}")

# Or list all CloudFormation stacks (not just RFT)
# all_stacks = list_rft_stacks(region=REGION, all_stacks=True)

## Step 3: Setup RFT Multiturn Infrastructure

Choose one deployment platform:
- **LOCAL**: Runs on your machine (development/testing)
- **ECS**: Serverless Fargate (production, recommended)
- **EC2**: Custom instances (more control)

In [ ]:
# Option 1: LOCAL (for development/testing)
rft_infra = RFTMultiturnInfrastructure(
    stack_name=STACK_NAME,
    region=REGION,
    python_venv_name=PYTHON_VENV_NAME,  # Required for LOCAL
    vf_env_id=VF_ENV_ID,
    # custom_env=custom_env, # Optional: specify custom environment
    infrastructure_arn=None,  # None = LOCAL; Recommended to use in SageMaker Notebooks
    # rft_role_name=RFT_ROLE_NAME,  # Optional: Provide an custom role name. If it doesn't exist, SDK will create an IAM role on your behalf
)

print(f"✅ Infrastructure initialized for {rft_infra.platform} platform")

In [ ]:
# Option 2: ECS (for production)
# Uncomment to use ECS:

# rft_infra = RFTMultiturnInfrastructure(
#     stack_name=STACK_NAME,
#     region=REGION,
#     vf_env_id=VF_ENV_ID,
#     infrastructure_arn="arn:aws:ecs:us-east-1:123456789012:cluster/my-ecs-cluster"  # TODO: Replace with your ECS cluster ARN
# )
#
# print(f"✅ Infrastructure initialized for {rft_infra.platform} platform")

In [ ]:
# Option 3: EC2
# Uncomment to use EC2:
#
# IMPORTANT: EC2 instance role must have the following permissions:
# - CloudFormation (CreateStack, DescribeStacks, etc.)
# - DynamoDB (CreateTable, PutItem, etc.)
# - IAM (CreateRole, PassRole, etc.)
# - Lambda (CreateFunction, UpdateFunctionCode, etc.)
# - SQS (CreateQueue, SendMessage, etc.)
# - S3 (CreateBucket, PutObject, GetObject for SAM and starter kit buckets)
# - CloudWatch Logs (CreateLogGroup, PutLogEvents, etc.)
# See notebook documentation section for complete IAM policy

# rft_infra = RFTMultiturnInfrastructure(
#     stack_name=STACK_NAME,
#     region=REGION,
#     python_venv_name=PYTHON_VENV_NAME,  # Required for EC2
#     vf_env_id=VF_ENV_ID,
#     infrastructure_arn="i-1234567890abcdef0"  # TODO: Replace with your EC2 instance ID or Instance ARN
# )
#
# print(f"✅ Infrastructure initialized for {rft_infra.platform} platform")

### Deploy Infrastructure

In [ ]:
# Deploy Lambda/SQS/DynamoDB infrastructure
rft_infra.setup()

print("✅ Infrastructure deployed")
print(f"   Lambda URL: {rft_infra.stack_outputs.proxy_function_url}")
print(f"   DynamoDB Table: {rft_infra.stack_outputs.dynamo_table_name}")

In [ ]:
# If queues have stale messages, flush them:
# rft_infra.flush_all_queues()

### Start Training Environment

In [ ]:
# Start training environment (reward function workers)
rft_infra.start_training_environment(vf_env_args={"use_think": False})

print("✅ Training environment started")

In [ ]:
rft_infra.get_logs(env_type=EnvType.TRAIN)

## Step 4: Prepare Your Dataset

RFT multiturn requires data in OpenAI format with multi-turn conversations.

In [ ]:
# Create sample RFT multiturn training data (Wordle format)
import json

# Base words for variety
words = [
    "beach",
    "crane",
    "light",
    "storm",
    "plant",
    "house",
    "water",
    "music",
    "dream",
    "space",
]

sample_data = [
    {
        "id": f"wordle_{i:03d}",
        "metadata": {
            "prompt": "Guess the 5-letter word",
            "answer": words[i % len(words)],
        },
    }
    for i in range(300)  # Generate 300 samples
]

# Save sample data locally
with open("rft_training_data.jsonl", "w") as f:
    for item in sample_data:
        f.write(json.dumps(item) + "\n")

print("✅ Sample data created: rft_training_data.jsonl")

### Load and Validate Dataset

In [ ]:
# Initialize dataset loader
loader = JSONLDatasetLoader(id="id", metadata="metadata")

# Load the data
loader.load("rft_training_data.jsonl")

# Preview the data
print("\n📊 Dataset Preview:")
loader.show(n=2)

In [ ]:
# Validate data for RFT multiturn
loader.validate(method=TrainingMethod.RFT_MULTITURN_LORA, model=Model.NOVA_LITE_2)

print("✅ Data validated for RFT multiturn training")

### Save Dataset to S3

In [ ]:
# Save to S3
train_path = loader.save_data(f"{S3_DATA_PATH}/train.jsonl")

print(f"\n✅ Training data saved to: {train_path}")

## Step 5: Configure HyperPod Runtime

RFT multiturn only supports SageMaker HyperPod (SMHP).

In [ ]:
# Setup HyperPod runtime
runtime = SMHPRuntimeManager(
    cluster_name=HYPERPOD_CLUSTER,
    namespace=NAMESPACE,
    instance_type="ml.p5.48xlarge",
    instance_count=2,
)

print("✅ Runtime configured for SageMaker HyperPod")
print(f"   Instance Type: {runtime.instance_type}")
print(f"   Instance Count: {runtime.instance_count}")

## Step 6: Initialize Nova Model Customizer

In [ ]:
# Create customizer with RFT multiturn infrastructure
customizer = NovaModelCustomizer(
    model=Model.NOVA_LITE_2,
    method=TrainingMethod.RFT_MULTITURN_LORA,  # or RFT_MULTITURN_FULL
    infra=runtime,
    data_s3_path=train_path,
    output_s3_path=S3_OUTPUT_PATH,
)

print("✅ NovaModelCustomizer initialized")
print(f"   Model: Nova Lite 2.0")
print(f"   Method: RFT Multiturn with LoRA")

## Step 7: Start Training

In [ ]:
# Start training
training_result = customizer.train(
    job_name="rft-multiturn-training",
    overrides={
        "max_steps": 10,
        "generation_replicas": 4,
        "max_new_tokens": 4096,
        "max_length": 16384,
        "global_batch_size": 64,
        "reasoning_effort": "null",
        "timeout": 1800,
    },
    rft_multiturn_infra=rft_infra,  # Pass RFT infrastructure
)

print(f"\n✅ Training job started: {training_result.job_id}")
training_result.dump()  # Save job info

## Step 8: Monitor Training Progress

In [ ]:
# Check training status
status, message = training_result.get_job_status()
print(f"Status: {status}")
print(f"Message: {message}")

In [ ]:
# View training logs
customizer.get_logs(limit=50)

In [ ]:
# View infrastructure logs
infra_logs = rft_infra.get_logs(limit=20)
print("\n📋 Infrastructure Logs:")
for log in infra_logs[-10:]:
    print(log)

In [ ]:
# Wait for training completion
print("Waiting for training to complete...")
while True:
    status, message = training_result.get_job_status()
    print(f"Status: {status}")

    if status == JobStatus.COMPLETED:
        print("\n✅ Training completed successfully!")
        print(f"Model artifacts: {training_result.model_artifacts.checkpoint_s3_path}")
        break
    elif status == JobStatus.FAILED:
        print(f"\n❌ Training failed: {message}")
        break

    time.sleep(60)  # Check every minute

## Step 9: Prepare for Evaluation

In [ ]:
# Stop training environment and start evaluation environment

rft_infra.kill_task(env_type=EnvType.TRAIN)
print("✅ Training environment stopped")

rft_infra.start_evaluation_environment(vf_env_args={"use_think": False})
print("✅ Evaluation environment started")

## Step 10: Run Evaluation

In [ ]:
# Create evaluation data
eval_data = [
    {
        "messages": [
            {"role": "user", "content": "What is Python?"},
            {"role": "assistant", "content": "Python is a programming language."},
            {"role": "user", "content": "What is it used for?"},
        ],
        "expected_response": "Python is used for web development, data science, and automation.",
    },
] * 10

with open("rft_eval_data.jsonl", "w") as f:
    for item in eval_data:
        f.write(json.dumps(item) + "\n")

# Load and validate
eval_loader = JSONLDatasetLoader(query="messages", response="expected_response")
eval_loader.load("rft_eval_data.jsonl")
eval_loader.validate(
    method=TrainingMethod.EVALUATION,
    model=Model.NOVA_LITE_2,
    eval_task=EvaluationTask.RFT_MULTITURN_EVAL,
)

# Save to S3
eval_path = eval_loader.save_data(f"{S3_DATA_PATH}/eval.jsonl")
print(f"✅ Evaluation data saved to: {eval_path}")

In [ ]:
# Setup HyperPod runtime
runtime = SMHPRuntimeManager(
    cluster_name=HYPERPOD_CLUSTER,
    namespace=NAMESPACE,
    instance_type="ml.p5.48xlarge",
    instance_count=1,
)

print("✅ Runtime configured for SageMaker HyperPod")
print(f"   Instance Type: {runtime.instance_type}")
print(f"   Instance Count: {runtime.instance_count}")

In [ ]:
# Create customizer with RFT multiturn eval
customizer = NovaModelCustomizer(
    model=Model.NOVA_LITE_2,
    method=TrainingMethod.EVALUATION,  # or RFT_MULTITURN_FULL
    infra=runtime,
    data_s3_path=train_path,
    output_s3_path=S3_OUTPUT_PATH,
)

print("✅ NovaModelCustomizer initialized")
print(f"   Model: Nova Lite 2.0")
print(f"   Method: RFT Multiturn with LoRA")

In [ ]:
# Run evaluation on trained model
eval_result = customizer.evaluate(
    job_name="rft-multiturn-eval",
    eval_task=EvaluationTask.RFT_MULTITURN_EVAL,
    model_path=training_result.model_artifacts.checkpoint_s3_path,
    rft_multiturn_infra=rft_infra,
)

print(f"\n✅ Evaluation started: {eval_result.job_id}")
eval_result.dump()

In [ ]:
# Wait for evaluation completion
print("Waiting for evaluation to complete...")
while True:
    status, message = eval_result.get_job_status()
    print(f"Status: {status}")

    if status == JobStatus.COMPLETED:
        print("\n✅ Evaluation completed!")
        eval_result.show()  # Display results
        break
    elif status == JobStatus.FAILED:
        print(f"\n❌ Evaluation failed: {message}")
        break

    time.sleep(30)

## Step 11: Cleanup Resources

In [ ]:
# Stop workers
rft_infra.kill_task(env_type=EnvType.EVAL)

# Clean up processes only (keeps CloudFormation stack and environment)
rft_infra.cleanup(delete_stack=False, cleanup_environment=False)

# Or clean up and delete environment (venv + starter kit for LOCAL/EC2, task definitions for ECS)
# rft_infra.cleanup(delete_stack=False, cleanup_environment=True)

# Or delete everything including CloudFormation stack
# rft_infra.cleanup(delete_stack=True, cleanup_environment=True)

---

## Summary

You've successfully:
1. ✅ Set up RFT multiturn infrastructure
2. ✅ Prepared multi-turn conversation data
3. ✅ Trained a Nova model with RFT multiturn
4. ✅ Evaluated the trained model
5. ✅ Cleaned up resources

## Next Steps

- Deploy your model to Bedrock for inference
- Experiment with different hyperparameters
- Try RFT_MULTITURN_FULL for full-rank training

## Key Methods Reference

```python
# Infrastructure
rft_infra.setup()                          # Deploy infrastructure
rft_infra.start_training_environment()     # Start training workers
rft_infra.start_evaluation_environment()   # Start eval workers
rft_infra.check_all_queues()              # Check queue health
rft_infra.flush_all_queues()              # Clear all queues
rft_infra.get_logs()                      # View worker logs
rft_infra.kill_task()                     # Stop workers
rft_infra.cleanup()                       # Clean up resources
```